# A notebook to construct a training/validation and test datasets from SOCOFing original dataset
# 1. Select images from specified sub directories (Real, Altered-Easy | Altered-Medium | Altered-Hard)
# 2. Sanitize images 
#    a. convert to greyscale then to RGB in order to match 3-channel input shape of pre-trained models
#    b. resize to (96, 96)
#    c. JPEG format
# 3. Save sanitized images to class-named subdirectories

In [1]:
# Import librairies
import os
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

## Explore the original dataset

In [2]:
# Constants for initial dataset

INPUT_ROOTDIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset"
DATASET_NAME = "SOCOFing"

# Sub-directories to scan for images
IMAGE_SUBDIRECTORIES = ['Real', 'Altered-Easy', 'Altered-Medium', 'Altered-Hard']
NUM_IMAGES_SUBDIRECTORIES = len(IMAGE_SUBDIRECTORIES)

DATASET_INPUT_DIR = INPUT_ROOTDIR + "/" + DATASET_NAME
OUTPUT_CSV_PATH = DATASET_INPUT_DIR + "_with_features.csv"

In [3]:
# List to store file information and images features in the original dataset
data = []

# Traverse the input directory to get all image files
for dirname, _, filenames in os.walk(DATASET_INPUT_DIR):
    for filename in filenames:

        if filename == ".DS_Store":  # Skip macOS system files
            continue
        
        # Extract the features from the dirname and the filename
        file_path = os.path.join(dirname, filename)
            
        # The individual is the name of the parent directory of the path
        individual = os.path.basename(os.path.dirname(file_path))

        if filename.find("Right") != -1:
            hand = "Right"
        elif filename.find("Left") != -1:
            hand = "Left"
        else:
           hand = None    

        if filename.find("thumb") != -1:
            finger = "thumb"
        elif filename.find("index") != -1:
            finger = "index"
        elif filename.find("middle") != -1:
            finger = "middle"
        elif filename.find("ring") != -1:
            finger = "ring"
        elif filename.find("little") != -1:
            finger = "little"
        else:
            finger = None

        # Extract the image features
        img = Image.open(file_path)
        try:
            img.verify()  # Verify that the file is a valid image
        except:
            print(f"Invalid image file: {file_path}")
            continue
        
        data.append([individual, file_path, img.format, img.size, img.mode, hand, finger])

# Create a DataFrame
df = pd.DataFrame(data, columns=["Individual", "ImagePath", "ImageFormat", "ImageSize", "ImageMode", "Hand", "Finger"])

# Add an index column
df.index.name = "ID"

# Save to CSV
df.to_csv(OUTPUT_CSV_PATH)

print(f"Full Organised Dataset CSV file with features saved at: {OUTPUT_CSV_PATH}")

Full Organised Dataset CSV file with features saved at: /Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_with_features.csv


In [4]:
# Display the information about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55270 entries, 0 to 55269
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Individual   55270 non-null  object
 1   ImagePath    55270 non-null  object
 2   ImageFormat  55270 non-null  object
 3   ImageSize    55270 non-null  object
 4   ImageMode    55270 non-null  object
 5   Hand         55270 non-null  object
 6   Finger       55270 non-null  object
dtypes: object(7)
memory usage: 3.0+ MB


In [ ]:
# Display the statistical summary of the dataset
df.describe()

In [ ]:
# show null values
df.isnull().sum()

In [5]:
# Display the distribution of individual, ImageFormat, ImageSize, ImageMode, hand, and fingers
print(df["Individual"].value_counts())
print(df["ImageFormat"].value_counts())
print(df["ImageSize"].value_counts())
print(df["ImageMode"].value_counts())
print(df["Hand"].value_counts())
print(df["Finger"].value_counts())

Individual
160__M    100
420__M    100
57__M     100
429__M    100
226__M    100
         ... 
127__F     72
120__M     72
442__F     68
94__M      65
16__M      61
Name: count, Length: 600, dtype: int64
ImageFormat
BMP    55270
Name: count, dtype: int64
ImageSize
(96, 103)     54837
(241, 298)      433
Name: count, dtype: int64
ImageMode
L       49270
RGBA     5956
RGB        44
Name: count, dtype: int64
Hand
Left     28038
Right    27232
Name: count, dtype: int64
Finger
thumb     11328
ring      11098
middle    11037
little    10931
index     10876
Name: count, dtype: int64


In [ ]:
# Distribution graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]] # For displaying purposes, pick columns that have between 1 and 50 unique values
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int((nCol + nGraphPerRow - 1) / nGraphPerRow)
    plt.figure(num = None, figsize = (6 * nGraphPerRow, 8 * nGraphRow), dpi = 80, facecolor = 'w', edgecolor = 'k')
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        columnDf = df.iloc[:, i]
        if (not np.issubdtype(type(columnDf.iloc[0]), np.number)):
            valueCounts = columnDf.value_counts()
            valueCounts.plot.bar()
        else:
            columnDf.hist()
        plt.ylabel('counts')
        plt.xticks(rotation = 90)
        plt.title(f'{columnNames[i]} (column {i})')
    plt.tight_layout(pad = 1.0, w_pad = 1.0, h_pad = 1.0)
    plt.show()

# show distribution of columns
plotPerColumnDistribution(df, 10, 5)

## Create the training/validation dataset

In [6]:
# Constants for sanitizing dataset

# traget images mode for conversion to pre-trained model
target_image_mode = "RGB"

# target image size for resizing
target_image_size = (128, 128)

# output format of the sanitized images
target_image_format = "JPEG"

In [7]:
# Constants for training/validation dataset

# Sub-directories to traverse
# SCAN_TRAINING_SUBDIRECTORIES = ['Real', 'Altered-Easy', 'Altered-Medium', 'Altered-Hard']
SCAN_TRAINING_SUBDIRECTORIES = ['Real', 'Altered-Easy']

# Define top-level directory and output CSV path
DATASET_TRAINING_OUTPUT_DIR = DATASET_INPUT_DIR + "_Training_CLAHE"
DATASET_TRAINING_OUTPUT_CSV_PATH = DATASET_TRAINING_OUTPUT_DIR + "_with_features.csv"

In [21]:
import cv2 

# function helper to create dataset
def create_dataset(input_dir: str, scan_dirs: tuple, output_dir: str, output_csv_path:str):

    # List to store file information
    imgdata = []

    print(f"Input directory : {input_dir}")
    print(f"Sub-directories to scan : {scan_dirs}")

    # Traverse the input directory to get all image files
    for dirname, _, filenames in os.walk(input_dir):
    
        # check whether image is in a sub-directory which ought to be considered
        found = False
        for dir in scan_dirs:
            if dirname.find(dir) != -1:
                found = True
                break
        if not found:
            continue

        for filename in filenames:

            if filename == ".DS_Store":  # Skip macOS system files
                continue
          
            file_path = os.path.join(dirname, filename)
            
            # Extract finger (label) from the filename
            if filename.find("thumb") != -1:
                finger = "thumb"
            elif filename.find("index") != -1:
                finger = "index"
            elif filename.find("middle") != -1:
                finger = "middle"
            elif filename.find("ring") != -1:
                finger = "ring"
            elif filename.find("little") != -1:
                finger = "little"
            else:
                print(f"Error extracting label from filename: {filename}")
                continue

            # Convert and resize the image
            #img = Image.open(file_path)
            img = cv2.imread(file_path)  # Read the image using OpenCV

            try:
                # first convert to grey scale
                gray_image = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

                #resize the image to match the input size of the model
                resized_gray_image = cv2.resize(gray_image, (128, 128), interpolation=cv2.INTER_LANCZOS4)

                # Apply CLAHE
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
                clahe_image = clahe.apply(resized_gray_image)

                # Convert the image back to RGB color space
                rgb_image_clahe = cv2.cvtColor(clahe_image, cv2.COLOR_GRAY2RGB)

            except Exception as e:
                print(f"Error processing image {file_path}: {e}")
                continue

            # Construct a non conlifting filename for the sanitized image
            filename = filename.replace(".BMP", f".{target_image_format.lower()}")
            output_sanitized_filename = os.path.join(output_dir, finger, filename)

            while os.path.exists(output_sanitized_filename):
                filename = filename.replace(f".{target_image_format.lower()}", f"_{np.random.randint(1000)}.{target_image_format.lower()}")
                output_sanitized_filename = os.path.join(output_dir, finger, filename)

            # save the image
            cv2.imwrite(output_sanitized_filename, rgb_image_clahe, [cv2.IMWRITE_JPEG_QUALITY, 95])  # Save the processed image temporarily

            try:
                # Re-open the saved image to get the final features after conversion and resizing
                img = Image.open(output_sanitized_filename)

                # Verify that the file is a valid image
                img.verify()
            except Exception as e:
                print(f"Error verifying image {output_sanitized_filename}: {e}")
                continue

            imgdata.append([output_sanitized_filename, img.format, img.size, img.mode, finger])

    # Create a DataFrame
    imgdf = pd.DataFrame(imgdata, columns=["ImagePath", "ImageFormat", "ImageSize", "ImageMode", "Finger"])

    # Add an index column
    imgdf.index.name = "ID"

    # Save to CSV
    imgdf.to_csv(output_csv_path)
    print(f"Output CSV file saved at: {output_csv_path}")

    return imgdf

In [22]:
imgdf = create_dataset(DATASET_INPUT_DIR, SCAN_TRAINING_SUBDIRECTORIES, DATASET_TRAINING_OUTPUT_DIR, DATASET_TRAINING_OUTPUT_CSV_PATH)

Input directory : /Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing
Sub-directories to scan : ['Real', 'Altered-Easy']
Output CSV file saved at: /Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Training_CLAHE_with_features.csv


In [23]:
# Display the information about the dataset
imgdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23931 entries, 0 to 23930
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ImagePath    23931 non-null  object
 1   ImageFormat  23931 non-null  object
 2   ImageSize    23931 non-null  object
 3   ImageMode    23931 non-null  object
 4   Finger       23931 non-null  object
dtypes: object(5)
memory usage: 934.9+ KB


In [24]:
# Display the statistical summary of the dataset
imgdf.describe()

,ImagePath,ImageFormat,ImageSize,ImageMode,Finger
count,23931,23931,23931,23931,23931
unique,23931,1,1,1,5
top,/Users/laurent/Projects/projet-fingerprint-val...,JPEG,"(128, 128)",RGB,index
freq,1,23931,23931,23931,4788


In [25]:
# show null values
imgdf.isnull().sum()

ImagePath      0
ImageFormat    0
ImageSize      0
ImageMode      0
Finger         0
dtype: int64

In [26]:
# Display the distribution of ImageFormat, ImageSize, ImageMode, and finger (label)
print(imgdf["ImageFormat"].value_counts())
print(imgdf["ImageSize"].value_counts())
print(imgdf["ImageMode"].value_counts())
print(imgdf["Finger"].value_counts())

ImageFormat
JPEG    23931
Name: count, dtype: int64
ImageSize
(128, 128)    23931
Name: count, dtype: int64
ImageMode
RGB    23931
Name: count, dtype: int64
Finger
index     4788
little    4788
middle    4788
thumb     4785
ring      4782
Name: count, dtype: int64


In [27]:
# show distribution of columns
plotPerColumnDistribution(imgdf, 10, 5)

NameError: name 'plotPerColumnDistribution' is not defined

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
print(f"Percentage of Thumbs: {imgdf['Finger'].value_counts().get('thumb', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Index fingers: {imgdf['Finger'].value_counts().get('index', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Middle fingers: {imgdf['Finger'].value_counts().get('middle', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Ring fingers: {imgdf['Finger'].value_counts().get('ring', 0) / len(imgdf) * 100:.2f}%")
print(f"Percentage of Little fingers: {imgdf['Finger'].value_counts().get('little', 0) / len(imgdf) * 100:.2f}%")

## Verify the training/validation dataset 

In [ ]:
# constants for the dataset

# Define the directory containing the images and the output CSV with labels
TRAINING_DATASET_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Training"
TRAINING_OUTPUT_CSV_PATH = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Training_with_labels.csv"

In [ ]:
def verify_dataset(input_dir: str, output_csv_path: str):

    # List to store file information
    sanimgdata = []

    print(f"Scanning dataset directory : {input_dir}")

    # Traverse the input directory to get all image files
    for dirname, _, filenames in os.walk(input_dir):
        for filename in filenames:

            if filename == ".DS_Store":  # Skip macOS system files
                continue

            # Ensure only image files are considered
            if filename.endswith(f".{target_image_format.lower()}"):

                file_path = os.path.join(dirname, filename)
            
                # Extract the label from the filename
                if filename.find("thumb") != -1:
                    finger = "thumb"
                elif filename.find("index") != -1:
                    finger = "index"
                elif filename.find("middle") != -1:
                    finger = "middle"
                elif filename.find("ring") != -1:
                    finger = "ring"
                elif filename.find("little") != -1:
                    finger = "little"
                else:
                    finger = None

                # verify the image
                img = Image.open(file_path)
                try:
                    img.verify()  # Verify that the file is a valid image
                except:
                    print(f"Invalid image file: {file_path}")
                    continue

                sanimgdata.append([file_path, finger])

    # Create a DataFrame
    sanimgdf = pd.DataFrame(sanimgdata, columns=["ImagePath", "Finger"])

    # Add an index column
    sanimgdf.index.name = "ID"

    # Save to CSV
    sanimgdf.to_csv(output_csv_path)
    print(f"Sanitized Dataset CSV file with labels saved at: {output_csv_path}")

    return sanimgdf

In [ ]:
sanimgdf = verify_dataset(TRAINING_DATASET_DIR, TRAINING_OUTPUT_CSV_PATH)

In [ ]:
# Display the information about the dataset
sanimgdf.info()

In [ ]:
# Display statistical summary of the dataset
sanimgdf.describe()

In [ ]:
# Display the distribution of fingers
print(sanimgdf["Finger"].value_counts())

In [ ]:
# show distribution of columns
plotPerColumnDistribution(imgdf, 10, 5)

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
print(f"Percentage of Thumbs: {sanimgdf['Finger'].value_counts().get('thumb', 0) / len(sanimgdf) * 100:.2f}%")
print(f"Percentage of Index fingers: {sanimgdf['Finger'].value_counts().get('index', 0) / len(sanimgdf) * 100:.2f}%")
print(f"Percentage of Middle fingers: {sanimgdf['Finger'].value_counts().get('middle', 0) / len(sanimgdf) * 100:.2f}%")
print(f"Percentage of Ring fingers: {sanimgdf['Finger'].value_counts().get('ring', 0) / len(sanimgdf) * 100:.2f}%")
print(f"Percentage of Little fingers: {sanimgdf['Finger'].value_counts().get('little', 0) / len(sanimgdf) * 100:.2f}%")

## Create the test dataset

In [ ]:
# Constants

# Sub-directories to traverse
# SCAN_TEST_SUBDIRECTORIES = ['Real', 'Altered-Easy', 'Altered-Medium', 'Altered-Hard']
SCAN_TEST_SUBDIRECTORIES = ['Real']

# Define top-level directory and output CSV path
DATASET_TEST_OUTPUT_DIR = DATASET_INPUT_DIR + "_Test"
OUTPUT_TEST_CSV_PATH = DATASET_TEST_OUTPUT_DIR + "_with_features.csv"

In [ ]:
imgdf = create_dataset(DATASET_INPUT_DIR, SCAN_TEST_SUBDIRECTORIES, DATASET_TEST_OUTPUT_DIR, OUTPUT_TEST_CSV_PATH)

In [ ]:
# Display the information about the dataset
imgdf.info()

In [ ]:
# Display the statistical summary of the dataset
imgdf.describe()

In [ ]:
# show null values
imgdf.isnull().sum()

In [ ]:
# Display the distribution of ImageFormat, ImageSize, ImageMode, and finger (label)
print(imgdf["ImageFormat"].value_counts())
print(imgdf["ImageSize"].value_counts())
print(imgdf["ImageMode"].value_counts())
print(imgdf["Finger"].value_counts())

In [ ]:
# show distribution of columns
plotPerColumnDistribution(imgdf, 10, 5)

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
if len(imgdf):
    print(f"Percentage of Thumbs: {imgdf['Finger'].value_counts().get('thumb', 0) / len(imgdf) * 100:.2f}%")
    print(f"Percentage of Index fingers: {imgdf['Finger'].value_counts().get('index', 0) / len(imgdf) * 100:.2f}%")
    print(f"Percentage of Middle fingers: {imgdf['Finger'].value_counts().get('middle', 0) / len(imgdf) * 100:.2f}%")
    print(f"Percentage of Ring fingers: {imgdf['Finger'].value_counts().get('ring', 0) / len(imgdf) * 100:.2f}%")
    print(f"Percentage of Little fingers: {imgdf['Finger'].value_counts().get('little', 0) / len(imgdf) * 100:.2f}%")

## Verify the test dataset

In [ ]:
# constants for the dataset

# Define the directory containing the images and the output CSV with labels
TEST_DATASET_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Test"
TEST_OUTPUT_CSV_PATH = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Test_with_labels.csv"

In [ ]:
sanimgdf = verify_dataset(TEST_DATASET_DIR, TEST_OUTPUT_CSV_PATH)

In [ ]:
# Display the information about the dataset
sanimgdf.info()

In [ ]:
# Display the statistical summary of the dataset
sanimgdf.describe()

In [ ]:
# show null values
sanimgdf.isnull().sum()

In [ ]:
# Display the distribution of finger (label)
print(sanimgdf["Finger"].value_counts())

In [ ]:
# show distribution of columns
plotPerColumnDistribution(sanimgdf, 10, 5)

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
if len(sanimgdf):
    print(f"Percentage of Thumbs: {sanimgdf['Finger'].value_counts().get('thumb', 0) / len(sanimgdf) * 100:.2f}%")
    print(f"Percentage of Index fingers: {sanimgdf['Finger'].value_counts().get('index', 0) / len(sanimgdf) * 100:.2f}%")
    print(f"Percentage of Middle fingers: {sanimgdf['Finger'].value_counts().get('middle', 0) / len(sanimgdf) * 100:.2f}%")
    print(f"Percentage of Ring fingers: {sanimgdf['Finger'].value_counts().get('ring', 0) / len(sanimgdf) * 100:.2f}%")
    print(f"Percentage of Little fingers: {sanimgdf['Finger'].value_counts().get('little', 0) / len(sanimgdf) * 100:.2f}%")